In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from typing import Iterable
from statsmodels.stats.multitest import multipletests

In [2]:
DATA_DIR  = "../../flatten/processed-data"
DATA_FILE = f"{DATA_DIR}/20251023-s288c-annotated-PPI-net.pkl"
nodes     = pd.read_pickle(DATA_FILE)

In [19]:
def extract_cellular_component_terms(path: str) -> Iterable[str]:
    """
    Extract controlled vocabulary terms from UniProt keywlist.txt file
    """
    terms = set()
    prefix = "HI   Cellular component:"
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if stripped.startswith(prefix):
                term = stripped[len(prefix):].strip()
                #print (term)
                if term.endswith("."):
                    term = term[:-1].strip()
                if term:
                    terms.add(term)
    return sorted(terms)

def analyze_localization_enrichment(nodes: pd.DataFrame, terms: Iterable[str]) -> pd.DataFrame:
    """
    For each controlled vocabulary term, test association between being a hub
    and being annotated with that term.

    2x2 table per term (restricted to nodes with any localization annotation):

                   has_term   lacks_term
        hubs           a          b
        non-hubs       c          d

    We compute:
      - two-sided odds_ratio (for effect size)
      - one-sided p-value for enrichment ("greater", hubs enriched for term)
      - one-sided p-value for depletion ("less", hubs depleted for term)
      - BH-FDR for each one-sided family (enriched_p, depleted_p)
    """

    # Work only on nodes that have *any* localization annotation
    nodes_annot = nodes[nodes["localization_keywords"].notna()].copy()

    loc = nodes_annot["localization_keywords"].astype(str)
    is_hub = nodes_annot["hub"].astype(bool)

    results = []

    for term in terms:
        # Boolean: node has this term (exact match within the ;-delimited list)
        has_term = loc.str.split(";").apply(
            lambda toks: any(t.strip() == term for t in toks)
        )

        a = int(((is_hub) & (has_term)).sum())       # hubs with term
        b = int(((is_hub) & (~has_term)).sum())      # hubs without term
        c = int(((~is_hub) & (has_term)).sum())      # non-hubs with term
        d = int(((~is_hub) & (~has_term)).sum())     # non-hubs without term

        table = [[a, b], [c, d]]

        # two-sided OR just for reporting
        odds_ratio, _ = fisher_exact(table, alternative="two-sided")

        # One-sided p-values
        # "greater": OR > 1 → term enriched in hubs
        enriched_p = fisher_exact(table, alternative="greater")[1]
        # "less": OR < 1 → term depleted in hubs
        depleted_p = fisher_exact(table, alternative="less")[1]

        results.append(
            {
                "cellular_component": term,
                "is_hub_has_term": a,
                "is_hub_lacks_term": b,
                "not_hub_has_term": c,
                "not_hub_lacks_term": d,
                "odds_ratio": odds_ratio,
                "enriched_p": enriched_p,
                "depleted_p": depleted_p,
            }
        )

    df = pd.DataFrame(results)

    # drop terms that never occur in any annotated node (a == 0 and c == 0)
    df = df[(df["is_hub_has_term"] + df["not_hub_has_term"]) > 0]

    if df.empty:
        return df  # nothing to test

    # Benjamini–Hochberg FDR for each directional test separately
    df["enriched_p_fdr"] = multipletests(df["enriched_p"].values, method="fdr_bh")[1]
    df["depleted_p_fdr"] = multipletests(df["depleted_p"].values, method="fdr_bh")[1]

    return df

In [20]:
# determine threshold for hubs based on quantile cutoff for degree centrality
cut          = 0.10
threshold    = nodes["degree_centrality"].quantile(1 - cut)

# add a boolean hub column: True if degree_centrality >= threshold, else False
nodes["hub"] = nodes["degree_centrality"] >= threshold
n_hubs       = nodes["hub"].sum()
n_nothubs    = (~nodes["hub"]).sum()

print(f"Threshold for defining a hub in MS data is {threshold:.5f}; there are {n_hubs} hubs and {n_nothubs} non hubs")

# filter out nodes with nan for localization_keywords
nodes        = nodes[~nodes["localization_keywords"].isna()]
print (f"{len(nodes)} nodes have at least one \"Cellular component term\"")

Threshold for defining a hub in MS data is 0.01248; there are 346 hubs and 3054 non hubs
3400 nodes have at least one "Cellular component term"


### Parse UniProt keyword list to get list of all "Cellular component" terms

In [24]:
# path to the controlled vocabulary file from UniProt
key_file     = "keywlist.txt"

# make set of the unique terms
terms        = extract_cellular_component_terms(key_file)
print (f"There are a total of {len(terms)} \"Cellular component\" terms in the UniProt controlled vocabulary")

# perform hypothesis tests and save results in a .csv file
results_df   = analyze_localization_enrichment(nodes, terms)

# quick view of the "Cellular compartment" terms in which hubs are enriched
display(results_df[(results_df["odds_ratio"] > 1.0) & (results_df["enriched_p_fdr"] < 0.05)].head(10))

# quick view of the "Cellular compartment" terms in which non-hubs are enriched
display(results_df[(results_df["odds_ratio"] < 1.0) & (results_df["depleted_p_fdr"] < 0.05)].head(10))

# save the results to csv
display(results_df.to_csv("cellular-compartment-associations.csv", index=False))

There are a total of 197 "Cellular component" terms in the UniProt controlled vocabulary


,cellular_component,is_hub_has_term,is_hub_lacks_term,not_hub_has_term,not_hub_lacks_term,odds_ratio,enriched_p,depleted_p,enriched_p_fdr,depleted_p_fdr
25,Cytoplasm,171,175,1161,1893,1.593223,2.875283e-05,0.999982,0.000144,1.0
95,Mitochondrion,55,291,222,2832,2.411071,3.175709e-07,1.000000,0.000005,1.0
103,Nucleus,176,170,1195,1859,1.610554,1.844387e-05,0.999989,0.000138,1.0


,cellular_component,is_hub_has_term,is_hub_lacks_term,not_hub_has_term,not_hub_lacks_term,odds_ratio,enriched_p,depleted_p,enriched_p_fdr,depleted_p_fdr


None

In [27]:
nodes["GO_terms"]

1       GO:0005737;GO:0005937;GO:0005886;GO:0000753;GO...
2       GO:0071944;GO:0005739;GO:0005886;GO:0005524;GO...
3       GO:0005801;GO:0005783;GO:0005789;GO:0000137;GO...
4       GO:0005737;GO:0005739;GO:0005634;GO:0004084;GO...
5       GO:0005759;GO:0005739;GO:0004084;GO:0052656;GO...
                              ...                        
3917                     GO:0005739;GO:0045277;GO:0006123
3920    GO:0000785;GO:0070775;GO:0005634;GO:0010484;GO...
3921                                GO:0005737;GO:0005634
3922                                GO:0005737;GO:0005634
3925    GO:0005829;GO:0005769;GO:0030139;GO:0012505;GO...
Name: GO_terms, Length: 3400, dtype: object